In [1]:
import os
os.chdir('../')
%pwd

'/Users/kiranprasadjp/Documents/Pros/NeuronWireTracingEngine'

In [2]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TorchConvertConfig:
    root_dir: Path
    train_data: Path
    label_data:Path
    params_box_size: list

In [3]:
from src.neuronTracer import *
from src.neuronTracer.constants import *
from src.neuronTracer.utils.common import read_yaml, create_directories

In [4]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_torch_config(self) -> TorchConvertConfig:
        config = self.config.torch_convert

        create_directories([config.root_dir])

        torch_config = TorchConvertConfig(
            root_dir= Path(config.root_dir),
            train_data= Path(config.train_data),
            label_data= Path(config.label_data),
            params_box_size= tuple(self.params.box_size_mac),
        )

        return torch_config

In [5]:
import torch
import zarr
import numpy as np

In [6]:
class TorchConvert:
    def __init__(self, config:TorchConvertConfig):
        self.config=config

    def extract_and_store_tensors(self):
        """
        Slices the Zarr warehouse into PyTorch Tensors with boundary safety.
        """
        images = zarr.open(self.config.train_data, mode='r')
        labels = zarr.open(self.config.label_data, mode='r')

        img_folder = os.path.join(self.config.root_dir, "images")
        lbl_folder = os.path.join(self.config.root_dir, "labels")
        os.makedirs(img_folder, exist_ok=True)
        os.makedirs(lbl_folder, exist_ok=True)

        # Get our (7, 64, 64) dimensions
        z_step, y_step, x_step = self.config.params_box_size
        z_max, y_max, x_max = images.shape

        counter = 0
        # The Triple Loop: The standard way to slice 3D volumes
        for z in range(0, z_max - z_step + 1, z_step):
            for y in range(0, y_max - y_step + 1, y_step):
                for x in range(0, x_max - x_step + 1, x_step):
                    
                    # Pull and convert
                    img_chunk = images[z:z+z_step, y:y+y_step, x:x+x_step]
                    lbl_chunk = labels[z:z+z_step, y:y+y_step, x:x+x_step]

                    # Convert to FloatTensor and add (C, Z, H, W) dimension
                    # PyTorch 3D CNNs expect 4D input: (Channel, Depth, Height, Width)
                    img_tensor = torch.from_numpy(img_chunk).float().unsqueeze(0)
                    lbl_tensor = torch.from_numpy(lbl_chunk).float().unsqueeze(0)

                    # Save using a padded index (e.g., box_0001.pt) for better folder sorting
                    torch.save(img_tensor, os.path.join(img_folder, f"box_{counter:04d}.pt"))
                    torch.save(lbl_tensor, os.path.join(lbl_folder, f"box_{counter:04d}.pt"))
                    
                    counter += 1

        logger.info(f"Successfully stored {counter} tensor pairs in {self.config.root_dir}")

In [7]:
try:
    config = ConfigurationManager()
    torch_config = config.get_torch_config()
    torch_convert = TorchConvert(config=torch_config)
    torch_convert.extract_and_store_tensors()
except Exception as e:
    raise e

[2026-03-22 17:28:43,754: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-03-22 17:28:43,756: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-22 17:28:43,757: INFO: common: created directory at: artifacts]
[2026-03-22 17:28:43,758: INFO: common: created directory at: artifacts/torch_convert]
[2026-03-22 17:28:54,819: INFO: 2310237170: Successfully stored 3584 tensor pairs in artifacts/torch_convert]
